# LangChain Tracer Context Reference

Developer-facing statements defined in `langchain_core.tracers.context`.

# `tracing_callback_var`

Legacy compatibility placeholder retained for code that imports the name but does not use it.

```python
tracing_callback_var: Any = None
```

---

# `tracing_v2_callback_var`

Context-local active LangSmith tracer.

```python
tracing_v2_callback_var: ContextVar[LangChainTracer | None] = ContextVar(
    "tracing_callback_v2",
    default=None,
)
```

`tracing_v2_enabled()` temporarily assigns a `LangChainTracer` to this variable and restores the previous context value when the context exits.

---

# `run_collector_var`

Context-local run collector.

```python
run_collector_var: ContextVar[RunCollectorCallbackHandler | None] = ContextVar(
    "run_collector",
    default=None,
)
```

`collect_runs()` temporarily assigns a `RunCollectorCallbackHandler` to this variable and restores the previous context value when the context exits.

The module registers this context variable as a non-inheritable configure hook during import.

---

# `tracing_v2_enabled`

Enables LangSmith tracing for LangChain runs created inside the context.

```python
@contextmanager
tracing_v2_enabled(
    project_name: str | None = None, # Project receiving traced runs; defaults through LangSmith when omitted
    *,
    example_id: str | UUID | None = None, # Optional example identifier associated with the tracer
    tags: list[str] | None = None, # Tags added to traced runs
    client: LangSmithClient | None = None, # Optional LangSmith client
) -> Generator[LangChainTracer, None, None] # Context manager yielding the configured tracer
```

A string `example_id` is converted to `UUID` before the tracer is created. The yielded `LangChainTracer` is installed in `tracing_v2_callback_var` for the duration of the context. The previous context value is restored in a `finally` block.

---

# `collect_runs`

Collects run traces created inside the context.

```python
@contextmanager
collect_runs(
) -> Generator[RunCollectorCallbackHandler, None, None] # Context manager yielding the active run collector
```

The yielded `RunCollectorCallbackHandler` is installed in `run_collector_var`. Its collected runs are available through `traced_runs`. The previous context value is restored in a `finally` block.

---

# `register_configure_hook`

Registers a context variable for callback-manager configuration.

```python
register_configure_hook(
    context_var: ContextVar[Any | None], # Context variable containing an optional callback handler
    inheritable: bool, # Whether child callback managers inherit the handler
    handle_class: type[BaseCallbackHandler] | None = None, # Handler class associated with an environment-variable toggle
    env_var: str | None = None, # Optional environment variable controlling handler creation
) -> None
```

The registration preserves the context variable, inheritance setting, optional handler class, and optional environment-variable name.

Raises `ValueError` when `env_var` is provided without a non-`None` `handle_class`.

In [ ]:
from langchain_core.runnables import RunnableLambda # Import a runnable that wraps a Python function
from langchain_core.tracers.context import collect_runs, run_collector_var # Import run collection tools


def square(number: int) -> int: # Define a function that will be traced
    return number * number # Return the square of the number


square_runnable = RunnableLambda(square) # Convert the function into a LangChain runnable

print("Before context:", run_collector_var.get()) # No collector is active initially

with collect_runs() as run_collector: # Start collecting runs inside this block
    print("Collector active:", run_collector_var.get() is run_collector) # Verify the active collector

    result = square_runnable.invoke(5) # Run the function and automatically create a trace

    print("Result:", result) # Display the function result
    print("Collected runs:", len(run_collector.traced_runs)) # Display the number of traces

    for run in run_collector.traced_runs: # Visit every collected run
        print("Run name:", run.name) # Display the run name
        print("Run type:", run.run_type) # Display the run type
        print("Inputs:", run.inputs) # Display the recorded inputs
        print("Outputs:", run.outputs) # Display the recorded outputs

print("After context:", run_collector_var.get()) # The active collector is removed